In [1]:
%load_ext autoreload
%autoreload 3
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import sys, math, warnings
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

sys.path.append('/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/spikeparam/spikeparam')
sys.path.append('/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/')
sys.path.append('/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/spikeparam/AP_empirical_paper1/datasets/spe-1/spe1_helper_modules/')
sys.path.append('/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/spikeparam/AP_empirical_paper1/datasets/shared_helper_modules')

from ridge_regression_utils import (
    load_cell_data, load_hpf_lfp_windows, build_ridge_matrices,
    run_ridge_regression, apply_fdr, plot_ridge_results,
    WAVEFORM_LABELS,
)

warnings.filterwarnings('ignore', message='use_inf_as_na option is deprecated',
                        category=FutureWarning, module='seaborn')
warnings.filterwarnings('ignore', message=r'invalid value encountered in log10',
                        category=RuntimeWarning, module=r'.*specparam.*')

In [2]:
# ── Cell configuration ────────────────────────────────────────────────────────
cell_num = 21

# Symmetric 50 ms windows immediately around the spike
PRE_WIN      = (-0.055, -0.005)  # pre-spike:  −55 → −5 ms
POST_WIN     = ( 0.005,  0.055)  # post-spike: +5 → +55 ms
BASELINE_WIN = (-0.20,  -0.10)   # baseline: −200 → −100 ms

# HPF detrending for LFP amp/std targets (0.1 Hz removes slow electrode drift)

# Ridge regression settings
ALPHAS          = np.logspace(-3, 3, 100)
N_PERM          = 1000
RNG_SEED        = 42
FORCE_RECOMPUTE = True   # True: rerun and overwrite pickle (needed after structure change)

In [3]:
# ── 1. Load data ──────────────────────────────────────────────────────────────
df_reg, specparam_by_spike, lfp_windows_by_spike = load_cell_data(cell_num)

# HPF windows for the HPF version (0.1 Hz removes slow electrode drift)
hpf_lfp = load_hpf_lfp_windows(cell_num, hpf_cutoff=0.1)[:len(specparam_by_spike)]

Found 63 chunk files. Assembling master list...
Loading Chunks: 100%|██████████| 63/63 [00:09<00:00,  6.47it/s]
Success! Master list assembled with 12552 total spikes.
c21: 12552 spikes loaded


In [4]:
# ── 2. Build matrices ─────────────────────────────────────────────────────────
# No HPF: raw LFP windows for amp/std targets
(X_waveform, X_log_isi, X_both, waveform_labels,
 Y_raw, target_names, target_labels) = build_ridge_matrices(
    df_reg, specparam_by_spike, lfp_windows_by_spike,
    pre_win=PRE_WIN, post_win=POST_WIN, baseline_win=BASELINE_WIN,
    hpf_lfp_by_spike=None,
)
# HPF 0.1 Hz: amp/std targets computed from high-pass filtered LFP trace
_, _, _, _, Y_hpf, _, _ = build_ridge_matrices(
    df_reg, specparam_by_spike, lfp_windows_by_spike,
    pre_win=PRE_WIN, post_win=POST_WIN, baseline_win=BASELINE_WIN,
    hpf_lfp_by_spike=hpf_lfp,
)
predictor_sets = {
    'Waveform only':      (X_waveform, waveform_labels),
    'Log ISI only':       (X_log_isi,  ['Log ISI']),
    'Waveform + Log ISI': (X_both,     waveform_labels + ['Log ISI']),
}

Target NaN %:
  Pre LFP Amp             0.0%
  Pre LFP Std             0.0%
  Pre Gamma AUC           0.0%
  Pre Exponent            0.0%
  Pre Theta AUC           0.0%
  Pre−BL LFP Amp          0.0%
  Pre−BL LFP Std          0.0%
  Pre−BL Gamma AUC        0.0%
  Pre−BL Exponent         0.0%
  Pre−BL Theta AUC        0.0%
  Post LFP Amp            0.0%
  Post LFP Std            0.0%
  Post Gamma AUC          0.0%
  Post Exponent           0.0%
  Post Theta AUC          0.0%
  Post−BL LFP Amp         0.0%
  Post−BL LFP Std         0.0%
  Post−BL Gamma AUC       0.0%
  Post−BL Exponent        0.0%
  Post−BL Theta AUC       0.0%
  Δ LFP Amp               0.0%
  Δ LFP Std               0.0%
  Δ Gamma AUC             0.0%
  Δ Exponent              0.0%
  Δ Theta AUC             0.0%
Target NaN %:
  Pre LFP Amp             0.0%
  Pre LFP Std             0.0%
  Pre Gamma AUC           0.0%
  Pre Exponent            0.0%
  Pre Theta AUC           0.0%
  Pre−BL LFP Amp          0.0%
  Pre−BL LF

In [5]:
# ── 3. Ridge regression ───────────────────────────────────────────────────────
import os, sys
sys.path.append('/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/spikeparam/AP_empirical_paper1/datasets/spe-1/spe1_helper_modules/')
from config import SPE1_PICKLE_ROOT

RIDGE_DIR     = os.path.join(SPE1_PICKLE_ROOT, 'ridge_regression_pickles')
save_path_raw = os.path.join(RIDGE_DIR, f'c{cell_num}_ridge_results.pkl')
save_path_hpf = os.path.join(RIDGE_DIR, f'c{cell_num}_ridge_results_hpf.pkl')

# No HPF — always recompute
results_raw = run_ridge_regression(
    Y_raw, predictor_sets, target_names,
    n_perm=N_PERM, rng_seed=RNG_SEED, alphas=ALPHAS,
    save_path=save_path_raw, force_recompute=FORCE_RECOMPUTE,
)
# HPF — load from cache if pickle exists (no need to rerun)
results_hpf = run_ridge_regression(
    Y_hpf, predictor_sets, target_names,
    n_perm=N_PERM, rng_seed=RNG_SEED, alphas=ALPHAS,
    save_path=save_path_hpf, force_recompute=False,
)

ridge CV: 100%|██████████| 75/75 [21:34<00:00, 17.26s/model, pred=Waveform +, target=delta_theta_]

  Saved: c21_ridge_results.pkl
  [cache] c21_ridge_results_hpf.pkl


In [6]:
# ── 4. FDR correction + save both versions ────────────────────────────────────
import pickle

for results, save_path in [(results_raw, save_path_raw), (results_hpf, save_path_hpf)]:
    results = apply_fdr(results, target_names, predictor_sets)
    with open(save_path, 'wb') as f:
        pickle.dump(results, f)
    print(f'Saved: {os.path.basename(save_path)}')

FDR (BH, q=0.05): 54/75 raw p<0.05 → 54/75 after correction

Target                         Waveform only          Log ISI only    Waveform + Log ISI
----------------------------------------------------------------------------------------
pre_lfp_amp               +0.1865* α=2.2e+02         +0.0722* α=23    +0.2876* α=1.9e+02
pre_lfp_std               +0.0786* α=3.3e+02      -0.0003  α=1e+03    +0.0791* α=3.8e+02
pre_gamma_auc               +0.0034* α=1e+03      -0.0003  α=1e+03      +0.0046* α=1e+03
pre_exponent                +0.0090* α=5e+02      -0.0007  α=1e+03      +0.0112* α=5e+02
pre_theta_auc               +0.0009* α=1e+03      -0.0003  α=1e+03      +0.0020* α=1e+03
prebc_lfp_amp             +0.0333* α=2.5e+02         +0.0629* α=11    +0.0872* α=2.2e+02
prebc_lfp_std             +0.0022* α=5.7e+02         +0.0079* α=81    +0.0110* α=6.6e+02
prebc_gamma_auc             -0.0007  α=1e+03      -0.0012  α=1e+03      -0.0013  α=1e+03
prebc_exponent              +0.0031* α=1e+03    +

## 2. Build Feature and Target Matrices

**Predictors (X)** — three competing sets:
- *Waveform only*: 8 spike shape features
- *Log ISI only*: 1 feature
- *Waveform + Log ISI*: 9 features combined

**Targets (Y)** — 25 LFP scalars (5 groups × 5 features):

| Group | Formula | Scientific question |
|-------|---------|-------------------|
| Pre absolute | mean(pre window) | LFP state when cell fires |
| Pre−BL | mean(pre) − mean(baseline) | LFP ramp into spike |
| Post absolute | mean(post window) | LFP state after spike |
| Post−BL | mean(post) − mean(baseline) | Spike-triggered response from baseline |
| Δ post−pre | mean(post) − mean(pre) | Net spike-triggered change |

Windows: Baseline −200→−100 ms  |  Pre −55→−5 ms  |  Post +5→+55 ms